# 📘 Assignment 05 — Conversational RAG with Memory

**Difficulty:** Intermediate → Advanced
**Prerequisite:** You must have completed **Assignment 04 — OpenAI Advanced RAG** and have the
`./chroma_db_advanced` vector store persisted on disk before starting this notebook.

---

## 🎯 Learning Objectives

By the end of this assignment you will be able to:

1. Explain why a stateless RAG pipeline fails in multi-turn conversations.
2. Manage conversation history in a structured, reusable way.
3. Rewrite follow-up questions into standalone queries using an LLM.
4. Build a history-aware retrieval pipeline from scratch.
5. Use LangChain's `ConversationalRetrievalChain` with `ConversationBufferWindowMemory`.
6. Implement a sliding-window token budget to keep prompts within context limits.

---

## 📋 Prerequisites

- ✅ Completed Assignment 04 (`OpenAI_Advanced_RAG_Assignment.ipynb`)
- ✅ `./chroma_db_advanced` folder exists (created by Assignment 04 — ChromaDB section)
- ✅ `OPENAI_API_KEY` set in a `.env` file in the project root
- ✅ All packages from Assignment 04 already installed


---
## ⚙️ Installation

Run this cell once to add the packages needed for this assignment.

In [ ]:
# Run this cell once
!pip install tiktoken langchain langchain-openai langchain-community chromadb -q


---
## 1 — Import Required Libraries

> ✅ This cell is complete — no changes needed.


In [ ]:
import os
import re
import warnings
from pathlib import Path
from typing import List, Dict, Any

# OpenAI + LangChain core
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv

# Vector store
from langchain_community.vectorstores import Chroma

# Conversational chains and memory
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferWindowMemory
from langchain_core.documents import Document

# Token counting
import tiktoken

warnings.filterwarnings("ignore")
print("✅ Libraries imported")


---
## 2 — Reload Setup from Assignment 04

> ✅ This cell is complete — it reconnects to the work you did in Assignment 04.

This section:
- Loads your OpenAI API key
- Re-creates the `llm` and `embeddings` objects
- **Loads the persisted ChromaDB vector store** (no re-indexing needed)
- Re-creates `base_retriever` so the rest of this notebook can use it


In [ ]:
# Load API key
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found. Add it to a .env file: OPENAI_API_KEY=sk-...")

# Recreate LLM and embeddings (identical to Assignment 04)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=api_key)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=api_key)

# Load the persisted vector store built in Assignment 04
vector_store = Chroma(
    collection_name="advanced_agentic_rag",
    embedding_function=embeddings,
    persist_directory="./chroma_db_advanced",
)

# Base retriever (same settings as Assignment 04)
base_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

print(f"✅ LLM     : {type(llm).__name__} ({llm.model_name})")
print(f"✅ Embeddings : {type(embeddings).__name__}")
print(f"✅ Vector store loaded from ./chroma_db_advanced")
print(f"✅ Base retriever ready (k=5)")


---
## 3 — Why Stateless RAG Breaks in Conversations

> ✅ No task in this section — read and run the demo to understand the problem.

A standard RAG pipeline treats every query as **independent**. This works well for
single questions but fails the moment a user asks a follow-up:

```
Turn 1: "How do you define a function in Python?"
Turn 2: "What about its performance characteristics?"  ← refers to "a function"
Turn 3: "Can you give me an example?"                  ← refers to "its performance"
```

Without memory, Turn 2 sends the phrase *"What about its performance characteristics?"*
directly to the retriever — which has no idea what "its" refers to and returns
irrelevant documents.

Run the cell below to see this failure in action.


In [ ]:
# Demo: stateless retrieval fails on follow-up questions
questions = [
    "How do you define a function in Python?",
    "What about its performance characteristics?",   # ← ambiguous without context
    "Can you give me an example?",                   # ← completely ambiguous
]

print("=== Stateless retrieval demo ===\n")
for i, q in enumerate(questions, 1):
    docs = base_retriever.invoke(q)
    top_doc = docs[0].page_content[:150].replace("\n", " ") if docs else "No docs found"
    print(f"Turn {i}: '{q}'")
    print(f"  → Top retrieved chunk: {top_doc}...")
    print()

print("Notice: Turn 2 and 3 retrieve unrelated content because the retriever")
print("has no memory of Turn 1.")


---
## 4 — Task 1: Build a Chat History Manager

**Goal:** Create a `ChatHistory` class that stores conversation turns and can format
them for injection into prompts.

**Why this matters:** Rather than scattering raw lists throughout your code, a
dedicated class gives you a clean API, makes token trimming easier to add later,
and keeps the rest of your code simple.

**Requirements — your class must implement:**

| Method | Signature | Behaviour |
|--------|-----------|-----------|
| `add_turn` | `(question: str, answer: str) -> None` | Appends one user/assistant exchange |
| `format_for_prompt` | `(max_turns: int = 5) -> str` | Returns the last `max_turns` exchanges as a human-readable string |
| `clear` | `() -> None` | Resets the history to empty |
| `__len__` | `() -> int` | Returns the number of stored turns |

**Hint — data structure:**
Store each turn as a dict `{"question": ..., "answer": ...}` in a list.
In `format_for_prompt`, iterate over the **last** `max_turns` entries and format
each as:
```
Human: <question>
Assistant: <answer>
```

**Hint — edge cases:**
- `format_for_prompt` should return an empty string `""` when history is empty.
- `max_turns` must never exceed the actual number of stored turns.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  🔧 YOUR TASK 1 of 6 — Build ChatHistory                           ║
# ╚══════════════════════════════════════════════════════════════════════╝

# add your code here


# ─── Self-test (run after you implement the class) ────────────────────────────
# Uncomment and run these lines to verify your implementation:
#
# history = ChatHistory()
# history.add_turn("What is Python?", "Python is a high-level programming language.")
# history.add_turn("Who created it?", "Guido van Rossum created Python in 1991.")
# assert len(history) == 2
# formatted = history.format_for_prompt()
# assert "What is Python?" in formatted
# assert "Guido van Rossum" in formatted
# history.clear()
# assert len(history) == 0
# print("✅ ChatHistory self-test passed")


---
## 5 — Task 2: Naive Conversational Agent

**Goal:** Build the simplest possible conversational agent by prepending the chat
history to the user's question before sending it to the existing RAG retriever.

**Why this matters:** Understanding *why* the naive approach fails is essential
before you build something better. This task forces you to observe the failure
mode directly.

**Requirements:**

1. Implement `ask_naive(question: str, history: ChatHistory) -> str` that:
   - Calls `history.format_for_prompt()` to get the formatted history string
   - Builds an augmented question:
     ```
     [Previous conversation:]
     <formatted history>

     [Current question:]
     <question>
     ```
   - Retrieves the top 3 documents from `base_retriever` using the **augmented** question
   - Builds a prompt combining the context + augmented question and calls `llm.invoke()`
   - Extracts the text from the response (`.content`) and returns it
   - Calls `history.add_turn(question, answer)` before returning

2. Run the three-turn demo below and write a brief comment in the reflection cell
   explaining what you observe.

**Hint — the retriever receives the full augmented string.** As the history grows,
the retriever query becomes long and unfocused — this is the core problem.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  🔧 YOUR TASK 2 of 6 — Naive Conversational Agent                  ║
# ╚══════════════════════════════════════════════════════════════════════╝

# add your code here


# ─── Three-turn demo (run after you implement ask_naive) ─────────────────────
# history_naive = ChatHistory()
# turns = [
#     "How do you define a function in Python?",
#     "What about its performance characteristics?",
#     "Can you give me an example?",
# ]
# for turn in turns:
#     print(f"User    : {turn}")
#     answer = ask_naive(turn, history_naive)
#     print(f"Agent   : {answer[:250]}\n")


---
## 6 — Task 3: Standalone Query Rewriter

**Goal:** Implement a function that uses the LLM to rewrite an ambiguous follow-up
question into a self-contained question that can be sent directly to the retriever.

**Example:**

```
History  : "How do you define a function in Python?"
Follow-up: "What about its performance characteristics?"
Rewritten: "What are the performance characteristics of Python functions?"
```

**Why this matters:** The retriever knows nothing about your conversation. Rewriting
the query is the cheapest, most reliable fix — it translates a context-dependent
question into one the retriever can answer correctly.

**Requirements — implement `rewrite_query`:**

```python
def rewrite_query(question: str, history: ChatHistory, llm) -> str:
    ...
```

- Format the last 3 turns from `history` as context
- Build a prompt that instructs the LLM to:
  - Use the conversation context to understand what the follow-up refers to
  - Output **only** the rewritten standalone question — no explanation, no preamble
- Call `llm.invoke(prompt)` and return `response.content.strip()`
- If history is empty, return the original `question` unchanged

**Hint — prompt structure that works well:**
```
Given the conversation history and a follow-up question, rewrite the follow-up
as a standalone question that can be understood without the history.
Output ONLY the rewritten question.

Conversation history:
<formatted history>

Follow-up question: <question>

Standalone question:
```


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  🔧 YOUR TASK 3 of 6 — Standalone Query Rewriter                   ║
# ╚══════════════════════════════════════════════════════════════════════╝

# add your code here


# ─── Self-test (run after you implement rewrite_query) ────────────────────────
# history_rw = ChatHistory()
# history_rw.add_turn(
#     "How do you define a function in Python?",
#     "You use the def keyword followed by the function name and parentheses."
# )
# rewritten = rewrite_query("What about its performance characteristics?", history_rw, llm)
# print(f"Original : What about its performance characteristics?")
# print(f"Rewritten: {rewritten}")
# # Should produce something like:
# # "What are the performance characteristics of Python functions?"


---
## 7 — Task 4: History-Aware Retrieval Pipeline

**Goal:** Combine the query rewriter (Task 3) with the RAG retrieval and generation
pipeline to build a clean, history-aware Q&A function — **without** using an agent.

**Pipeline:**
```
question + history
       ↓
   rewrite_query()          ← Task 3
       ↓
  base_retriever.invoke()   ← retrieves on the REWRITTEN query
       ↓
  build prompt with:
    - retrieved context
    - original conversation history (last 5 turns)
    - original question           ← the user's actual words
       ↓
  llm.invoke()
       ↓
  history.add_turn()
       ↓
  return answer
```

**Requirements — implement `ask_history_aware`:**

```python
def ask_history_aware(question: str, history: ChatHistory) -> str:
    ...
```

1. Rewrite the question using `rewrite_query()`; print the rewritten query so you
   can verify it is resolving references correctly.
2. Retrieve the top 5 documents using the **rewritten** query.
3. Build a context string from the retrieved document contents (first 300 chars each).
4. Build a final prompt containing:
   - The retrieved context
   - The formatted conversation history (last 5 turns)
   - The **original** question (not the rewritten one — you want the answer to
     match the user's phrasing)
5. Call `llm.invoke(final_prompt)` and extract `.content`.
6. Call `history.add_turn(question, answer)` before returning.

**Hint — suggested final prompt template:**
```
You are a helpful assistant. Answer the question based on the context below.

Context from knowledge base:
<retrieved context>

Conversation history:
<formatted history>

Question: <original question>
Answer:
```


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  🔧 YOUR TASK 4 of 6 — History-Aware Retrieval Pipeline            ║
# ╚══════════════════════════════════════════════════════════════════════╝

# add your code here


# ─── Three-turn demo (run after you implement ask_history_aware) ──────────────
# history_aware = ChatHistory()
# turns = [
#     "How do you define a function in Python?",
#     "What about its performance characteristics?",
#     "Can you give me an example?",
# ]
# for turn in turns:
#     print(f"User    : {turn}")
#     answer = ask_history_aware(turn, history_aware)
#     print(f"Agent   : {answer[:250]}\n")


---
## 8 — Task 5: LangChain ConversationalRetrievalChain

**Goal:** Replace your hand-built pipeline (Task 4) with LangChain's built-in
`ConversationalRetrievalChain`, which handles query rewriting and memory
management for you.

**Why this matters:** Once you understand the problem (Tasks 1–4), using the
framework's abstractions saves time and reduces bugs in production.

**Requirements:**

1. Create a `ConversationBufferWindowMemory` object:
   - `k=5` — keep the last 5 conversation turns
   - `memory_key="chat_history"` — the key the chain uses to inject history
   - `return_messages=True` — return `HumanMessage`/`AIMessage` objects
   - `output_key="answer"` — tells memory which chain output to store

2. Create a `ConversationalRetrievalChain` using `.from_llm()`:
   - Pass `llm`, `base_retriever`, and `memory`
   - Set `return_source_documents=True` so you can inspect what was retrieved
   - Set `verbose=True` to observe the internal query rewriting step

3. Invoke the chain with `chain.invoke({"question": question})`.
   The chain returns a dict — `result["answer"]` is the generated response.

4. Run the same three-turn demo as Tasks 2 and 4 and compare the output quality.

**Hint — `ConversationBufferWindowMemory` manages history automatically.**
You do not need to call `add_turn()` manually — the chain writes to memory itself.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  🔧 YOUR TASK 5 of 6 — ConversationalRetrievalChain                ║
# ╚══════════════════════════════════════════════════════════════════════╝

# Step 1 — Create memory
# memory = ConversationBufferWindowMemory(...)

# add your code here


# Step 2 — Create the chain
# conv_chain = ConversationalRetrievalChain.from_llm(...)

# add your code here


# ─── Three-turn demo (run after you create the chain) ────────────────────────
# turns = [
#     "How do you define a function in Python?",
#     "What about its performance characteristics?",
#     "Can you give me an example?",
# ]
# for turn in turns:
#     print(f"User : {turn}")
#     result = conv_chain.invoke({"question": turn})
#     print(f"Agent: {result['answer'][:250]}")
#     print(f"  (retrieved {len(result['source_documents'])} docs)\n")


---
## 9 — Task 6: Sliding Window with Token Budget

**Goal:** Implement a function that trims conversation history to fit within a token
budget, keeping the most recent turns and discarding older ones when the limit is hit.

**Why this matters:** LLMs have a finite context window (e.g., 128 k tokens for
GPT-4o). In a long conversation, the combined history + retrieved context +
system prompt can exceed this limit, causing API errors or silent truncation.
A token budget ensures your application stays within bounds.

**Requirements — implement `trim_history_to_token_budget`:**

```python
def trim_history_to_token_budget(
    history: List[Dict[str, str]],
    max_tokens: int = 1000,
    model: str = "gpt-4o-mini",
) -> List[Dict[str, str]]:
    ...
```

Where `history` is a list of `{"question": ..., "answer": ...}` dicts (as stored
by your `ChatHistory` class).

**Algorithm:**

1. Get the token encoder: `enc = tiktoken.encoding_for_model(model)`
2. Walk the history **from newest to oldest** (reverse order).
3. For each turn, count tokens in `question + answer` using `len(enc.encode(text))`.
4. Keep adding turns to a `kept` list as long as the running total stays ≤ `max_tokens`.
5. Stop as soon as the budget would be exceeded.
6. Always keep at least the **last 1 turn**, even if it exceeds the budget alone.
7. Return `kept` **reversed** back to chronological order.

**Hint — token counting:**
```python
enc = tiktoken.encoding_for_model("gpt-4o-mini")
n_tokens = len(enc.encode("some text here"))
```

**Hint — testing your function:**
Create a `ChatHistory` with 10+ turns and call `trim_history_to_token_budget`
with a small budget (e.g., 200 tokens). Verify that the result contains fewer
turns and that the most recent turns are preserved.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  🔧 YOUR TASK 6 of 6 — Sliding Window with Token Budget            ║
# ╚══════════════════════════════════════════════════════════════════════╝

# add your code here


# ─── Self-test (run after you implement trim_history_to_token_budget) ─────────
# # Build a history with 8 turns
# test_history = ChatHistory()
# for i in range(8):
#     test_history.add_turn(
#         f"Question number {i+1}: tell me something interesting about Python feature {i+1}.",
#         f"Answer number {i+1}: Python feature {i+1} is noteworthy because of its design."
#     )
#
# raw = [{"question": t["question"], "answer": t["answer"]} for t in test_history._turns]
# # NOTE: adjust ._turns to match whatever internal attribute name you used
#
# trimmed = trim_history_to_token_budget(raw, max_tokens=200, model="gpt-4o-mini")
# print(f"Original turns : {len(raw)}")
# print(f"Trimmed turns  : {len(trimmed)}")
# print(f"Most recent kept: {trimmed[-1]['question'][:60]}")
# assert len(trimmed) < len(raw), "Expected fewer turns after trimming"
# assert trimmed[-1] == raw[-1], "Most recent turn must always be kept"
# print("✅ trim_history_to_token_budget self-test passed")


---
## 10 — Side-by-Side Comparison

> ✅ This cell is complete — run it after completing all 6 tasks.

This section runs the same three-turn conversation through all four approaches
(Tasks 2, 4, 5, and a token-trimmed variant of Task 4) and prints the answers
side by side so you can compare quality directly.


In [ ]:
# Side-by-side comparison — run after completing Tasks 1-6

TURNS = [
    "How do you define a function in Python?",
    "What about its performance characteristics?",
    "Can you give me an example?",
]

approaches = {
    "Task 2 — Naive": None,        # fill in your function/history object
    "Task 4 — History-Aware": None, # fill in your function/history object
    "Task 5 — LangChain Chain": None,  # fill in your chain
}

# ── Uncomment and adapt after completing all tasks ────────────────────────────
# results = {name: [] for name in approaches}
#
# # Task 2
# h2 = ChatHistory()
# for t in TURNS:
#     results["Task 2 — Naive"].append(ask_naive(t, h2)[:200])
#
# # Task 4
# h4 = ChatHistory()
# for t in TURNS:
#     results["Task 4 — History-Aware"].append(ask_history_aware(t, h4)[:200])
#
# # Task 5  (memory resets each time you recreate the chain — recreate it here)
# mem5 = ConversationBufferWindowMemory(
#     k=5, memory_key="chat_history", return_messages=True, output_key="answer"
# )
# chain5 = ConversationalRetrievalChain.from_llm(
#     llm=llm, retriever=base_retriever, memory=mem5,
#     return_source_documents=True, verbose=False,
# )
# for t in TURNS:
#     results["Task 5 — LangChain Chain"].append(
#         chain5.invoke({"question": t})["answer"][:200]
#     )
#
# # Print comparison table
# for turn_idx, turn in enumerate(TURNS):
#     print(f"\n{'='*90}")
#     print(f"Turn {turn_idx+1}: {turn}")
#     print('='*90)
#     for name, answers in results.items():
#         print(f"  [{name}]")
#         print(f"  {answers[turn_idx]}\n")
print("Uncomment the code above after completing Tasks 1-6.")


---
## 11 — Reflection

Answer these questions directly in this cell by replacing each `_your answer here_`
placeholder:

| Question | Your observation |
|----------|-----------------|
| Which approach (Tasks 2, 4, or 5) gave the most accurate answers on Turn 2 and 3, and why? | _your answer here_ |
| What is the main risk of the naive approach (Task 2) in a long conversation? | _your answer here_ |
| `ConversationBufferWindowMemory(k=5)` discards turns older than 5. When is this a problem? | _your answer here_ |
| In Task 6, you implemented a token-budget trimmer. Name one scenario where a token budget is strictly necessary. | _your answer here_ |
| If you had to deploy a conversational RAG system for a customer-support chatbot, which approach would you choose and what would you add? | _your answer here_ |
